# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaant7/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Type: Classification (binary)

I'm framing this as a binary classification task: predict is_declining = (trend_direction == "down") from observed signals (impressions, position, CTR, freshness, word count, engagement).

I'm choosing classification because trend_direction is already a categorical label in the data, which makes it a natural target. But the classifier itself isn't the end goal — the point is to use it as a tool for Lane 1's real question: which signals actually carry information about movement. A trained classifier's coefficients or feature importances let me measure that systematically, instead of eyeballing individual correlations one at a time.

I'm not choosing clustering, because that would group pages by similarity without reference to a specific outcome — useful for a different lane, but not for answering "which signals predict movement." I'm not choosing ranking or scoring yet either, because those are about producing a prioritized action list (Lane 2/Lane 4's job) — I'm still one step earlier, at signal discovery.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

What I would predict: is_declining = (trend_direction == "down") — a binary label indicating whether a page's trend is currently downward.

Where the label comes from: This is a defined rule / proxy label, not a genuinely observed future outcome. trend_direction is a bucket calculated from the current 90-day window (via trend_pct), not something measured after a clearly defined decision point. It tells me the page's current trajectory, but it is not the same as "this page will decline over the next 30 days" — that would require a future-looking window I haven't built.

Note on the label's structure: trend_direction actually has three categories — down, stable, and up — not just two. My binary label groups "stable" and "up" together as "not declining." This is a simplification worth stating explicitly, since a stable page and a rising page aren't really the same thing; a future version of this label could separate them into three classes, or focus specifically on "down vs. up" and treat "stable" pages differently.

I'm using this proxy because Week 2 is about framing the task, not building the final model. A stronger, more honest label for a later stage would follow the shape: features from a prior window → outcome in a following window (e.g. prior 90 days → decline over the next 30 days) — which avoids using information calculated from the same window as the features.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: ROC AUC

I'm choosing ROC AUC over plain accuracy because the classes are close to balanced but not perfectly so (~54% declining) — a model that just predicts "declining" every time would score ~54% accuracy without learning anything. ROC AUC instead measures how well the model separates declining from non-declining pages, regardless of class balance, which is a more honest test of whether the signals carry real information.

What "good" means here: An AUC meaningfully above 0.5 (random guessing) shows the signals genuinely relate to trend movement. Using the starter pipeline's own benchmark as a reference point — baseline rules score 0.627, and a random forest reaches 0.750 — I'll treat anything in that range as evidence the signals matter, and anything close to 0.5 as evidence they don't.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: One row = one content item (content_id), aggregated over a 90-day observation window. Each row represents a single page's summarized performance — not a daily snapshot, not a client, not a query.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Apply the same feature-prep rules from Week 1
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates(subset="content_id")

print("Shape:", df.shape)
print("Unique content_ids:", df["content_id"].nunique())
print("Rows == unique content_ids?", len(df) == df["content_id"].nunique())

# Show a few representative columns to prove one row = one page
df[["content_id", "impressions_90d", "avg_position", "ctr",
    "content_age_days", "word_count", "trend_direction"]].head()

Shape: (30000, 44)
Unique content_ids: 30000
Rows == unique content_ids? True


,content_id,impressions_90d,avg_position,ctr,content_age_days,word_count,trend_direction
0,content_304f48230142,3803,10.6,0.76,187,3221.0,down
1,content_a1fb4e703a9e,15320,20.3,0.05,445,2481.0,down
2,content_9aa793d4d895,12581,36.5,0.09,141,3515.0,down
3,content_331d6c4de07b,11751,6.2,0.49,463,NaN,stable
4,content_d99b7a2d90ca,19140,44.0,0.13,263,2803.0,down


Confirmed: 30,000 rows, 30,000 unique content_id values — one row per page, no duplicates. trend_direction has three values (down/stable/up); my binary target treats "down" as the positive class.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (like "flag if impressions > 500 and CTR < 0.5") only looks at one or two signals at a time, with thresholds a person picked by hand. But the signals in this data barely relate to each other in a clean, linear way — the CTR-position correlation I found in Week 1 was only -0.073, a very weak relationship. Real decline shows up through combinations of moderate signals, not one obvious red flag: a page with moderate impressions, aging content, and weak engagement can be just as much a priority as one page with one extreme signal.

This isn't just theoretical — the starter pipeline shows the gap directly. The hand-written baseline rule scores Precision@50 = 0.240 (about 12 correct picks out of its top 50), while a random forest reaches 0.740 (about 37 correct picks out of 50) — roughly a 3x improvement. That gap is exactly what "the pattern is too messy for an if-statement" means in practice: a model that can weigh several signals together, and learn how much each one matters, finds meaningfully better priorities than a person's best guess at a threshold.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [] Every section above is filled — markdown thinking AND the code that backs it
- [] The notebook runs top to bottom with no errors (Runtime → Run all)
- [] No client names, URLs, or private queries anywhere
- [] My claims use careful words: observed, measured, directional, decision-support
- [] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.